# Building thermal modelling — Question 3: how good are the forecasts?

In Question 2 we built three R-C models (A, B, C) and fitted them on **year 1**. Now we test them on **year 2** — data
the models have never seen — and answer the last question of the task:

> **How well do the models predict the indoor temperature in the two required test windows, across different forecast
> horizons, and which model is best — and why?**

The two windows are fixed by the task:

- **Winter:** the first 10 days of year 2 (days 1–10).
- **Summer:** 10 days starting on day 200 of year 2 (days 200–209).

This is a standalone notebook: it reloads the data, rebuilds and refits the three models (a few seconds), then evaluates
them. Figures are saved to `../figures/`.

## Outline

1. [Setup and data](#sec-setup)
2. [Rebuild and fit the three models on year 1](#sec-fit)
3. [The two test windows](#sec-windows)
4. [Two ways to predict: one-step vs. multi-step open-loop](#sec-modes)
5. [How we score the forecasts (RMSE, MAE, bias)](#sec-metrics)
6. [Error vs. forecast horizon](#sec-curves)
7. [Forecast vs. reality: the 10-day open-loop run](#sec-freerun)
8. [Residuals: size, bias, and what the models miss](#sec-resid)
9. [Summary table](#sec-table)
10. [Answer: which model is best, and why](#sec-answer)

<a id="sec-setup"></a>
## 1. Setup and data

In [13]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name, h=420):
    # save an interactive .html and a static .png, and return the figure so it shows in the notebook
    fig.update_layout(height=h, margin=dict(l=60, r=30, t=70, b=50))
    fig.write_html(f"{FIG_DIR}/{name}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(f"{FIG_DIR}/{name}.png", scale=2)
    except Exception as e:
        print(f"[warn] could not save PNG for {name}: {e}")
    return fig

# load data, split the two years by row position, add °C columns
df = pd.read_csv("../data/interview_data_buildings/interview_data_buildings.csv", index_col=0)
df["year"] = np.where(np.arange(len(df)) < 35040, 1, 2)
for col in ["indoor_temperature", "outside_temperature"]:
    df[col + "_C"] = df[col] - 273.15
print("Loaded", len(df), "rows")

Loaded 70080 rows


<a id="sec-fit"></a>
## 2. Rebuild and fit the three models on year 1

These are exactly the models from Question 2 (1R1C, 2R2C, 3R2C), fitted on year 1 with the same multiple-shooting method. The code is repeated here so this notebook runs on its own; see the Question 2 notebook for the full explanation of each model and the fitting choices.

In [14]:
from scipy.linalg import expm
from scipy.optimize import least_squares
from numba import njit

# --- training data: year 1, 15-minute steps ---
d1 = df[df.year == 1]
Ti_tr = d1["indoor_temperature_C"].values
src_tr = {"To": d1["outside_temperature_C"].values, "Qh": d1["heating_power"].values,
          "Qs": d1["global_horizontal_solar_radiation"].values}
DT = 900.0                      # 15 min
HSET = [2, 96, 288]             # multiple-shooting windows used for fitting: 30 min, 1 day, 3 days

def matrices(model, p):
    # continuous-time A (state) and B (input) matrices for each model
    if model == "A":
        Ri, Ci = p["Ri"], p["Ci"]
        A = np.array([[-1/(Ri*Ci)]]); B = np.array([[1/(Ri*Ci), 1/Ci]]); cols = ("To", "Qh")
    elif model == "B":
        Rie, Rea, Ci, Ce = p["Rie"], p["Rea"], p["Ci"], p["Ce"]
        A = np.array([[-1/(Rie*Ci), 1/(Rie*Ci)], [1/(Rie*Ce), -1/(Rie*Ce) - 1/(Rea*Ce)]])
        B = np.array([[0.0, 1/Ci], [1/(Rea*Ce), 0.0]]); cols = ("To", "Qh")
    else:
        Rie, Ria, Rea, Ci, Ce, As = p["Rie"], p["Ria"], p["Rea"], p["Ci"], p["Ce"], p["As"]
        A = np.array([[-1/(Rie*Ci) - 1/(Ria*Ci), 1/(Rie*Ci)], [1/(Rie*Ce), -1/(Rie*Ce) - 1/(Rea*Ce)]])
        B = np.array([[1/(Ria*Ci), 1/Ci, As/Ci], [1/(Rea*Ce), 0.0, 0.0]]); cols = ("To", "Qh", "Qs")
    return A, B, cols

def discretize(A, B, dt):
    # exact zero-order-hold step: F = exp(A dt), G = A^{-1}(F-I)B
    F = expm(A * dt)
    try:
        G = np.linalg.solve(A, (F - np.eye(A.shape[0])) @ B)
    except np.linalg.LinAlgError:
        I = np.eye(A.shape[0]); S = I*dt; term = I*dt
        for k in range(1, 12):
            term = term @ A * dt / (k+1); S = S + term
        G = S @ B
    return F, G

@njit(cache=False)
def _freerun(F, G, U, x0, n, ns):
    X = np.zeros((n, ns)); X[0] = x0
    for t in range(1, n):
        X[t] = F @ X[t-1] + G @ U[t-1]
    return X

@njit(cache=False)
def _ms_resid(F, G, U, Tm, H, n, ns):
    res = np.zeros(n); nseg = n // H
    for sg in range(nseg):
        s = sg*H; x = np.zeros(ns)
        for k in range(ns): x[k] = Tm[s]
        for t in range(1, H):
            x = F @ x + G @ U[s+t-1]; res[s+t] = x[0] - Tm[s+t]
    return res

PARAM_NAMES = {"A": ["Ri", "Ci"], "B": ["Rie", "Rea", "Ci", "Ce"],
               "C": ["Rie", "Ria", "Rea", "Ci", "Ce", "As"]}
P0 = {"A": dict(Ri=4e-3, Ci=1.5e7), "B": dict(Rie=8e-4, Rea=4e-3, Ci=8e6, Ce=4e7),
      "C": dict(Rie=8e-4, Ria=6e-3, Rea=4e-3, Ci=8e6, Ce=4e7, As=4.0)}
R_LO, R_HI = np.log(5e-4), np.log(3e-2); C_LO, C_HI = np.log(1e6), np.log(1e8); AS_LO, AS_HI = np.log(0.01), np.log(30.0)
def _isR(n): return n in ("Ri", "Rie", "Ria", "Rea")
def _isC(n): return n in ("Ci", "Ce")
def vec_to_params(model, z):
    # the optimiser works with log-parameters, so everything stays positive
    return {nme: np.exp(z[i]) for i, nme in enumerate(PARAM_NAMES[model])}
def params_to_vec(model, p):
    return np.array([np.log(p[nme]) for nme in PARAM_NAMES[model]], float)
def bounds(model):
    lo, hi = [], []
    for nme in PARAM_NAMES[model]:
        if   _isR(nme):   lo.append(R_LO);  hi.append(R_HI)
        elif _isC(nme):   lo.append(C_LO);  hi.append(C_HI)
        else:             lo.append(AS_LO); hi.append(AS_HI)   # the solar window area
    return np.array(lo), np.array(hi)
def U_of(cols, src): return np.ascontiguousarray(np.column_stack([src[c] for c in cols]))

def fit(model):
    A0, _, cols = matrices(model, P0[model]); ns = A0.shape[0]; U = U_of(cols, src_tr)
    def resid(z):
        A, B, _ = matrices(model, vec_to_params(model, z)); F, G = discretize(A, B, DT)
        return np.concatenate([_ms_resid(F, G, U, Ti_tr, H, len(Ti_tr), ns) for H in HSET])
    lo, hi = bounds(model)
    sol = least_squares(resid, params_to_vec(model, P0[model]), bounds=(lo, hi),
                        method="trf", x_scale="jac", max_nfev=4000)
    return vec_to_params(model, sol.x)

FITS = {m: fit(m) for m in ["A", "B", "C"]}     # fit all three (a few seconds)
print("Fitted models A, B, C on year 1.")

Fitted models A, B, C on year 1.


<a id="sec-windows"></a>
## 3. The two test windows

We pull out the exact windows the task asks for, from **year 2**. Each is 10 days = 960 fifteen-minute steps.

In [15]:
def get_window(day_lo, day_hi):
    d = df[(df.year == 2) & (df.day_of_year >= day_lo) & (df.day_of_year < day_hi)]
    src = {"To": d.outside_temperature_C.values, "Qh": d.heating_power.values,
           "Qs": d.global_horizontal_solar_radiation.values}
    return d, d.indoor_temperature_C.values, src

WINDOWS = {"Winter (year 2, days 1–10)":   (1, 11),
           "Summer (year 2, days 200–209)": (200, 210)}
for name, (lo, hi) in WINDOWS.items():
    d, meas, _ = get_window(lo, hi)
    print(f"{name}: {len(meas)} steps | indoor {meas.min():.1f}–{meas.max():.1f} °C | "
          f"outdoor {d.outside_temperature_C.min():.1f}–{d.outside_temperature_C.max():.1f} °C")

Winter (year 2, days 1–10): 960 steps | indoor 17.7–23.0 °C | outdoor 0.6–11.3 °C
Summer (year 2, days 200–209): 960 steps | indoor 20.4–27.7 °C | outdoor 10.3–30.9 °C


<a id="sec-modes"></a>
## 4. Two ways to predict: one-step vs. multi-step open-loop

There are two very different ways to ask a model "what will the indoor temperature be?", and they matter for
different real uses:

- **One-step prediction (15 minutes ahead).** At every moment we give the model the *latest measured* temperature and
  ask only for the next 15 minutes. This is the setting of a system that always has fresh sensor readings (for example a
  state estimator). For the two-node models we keep the measured indoor temperature but let the model carry its own
  estimate of the hidden wall temperature forward — a simple observer.

- **Multi-step open-loop prediction.** We start the model once from a known state and then let it **run forward on its
  own predictions**, feeding it only the future weather and heating, with **no measurement feedback**. The forecast for
  hour 2 is built on the model's own (possibly wrong) guess for hour 1, so errors can accumulate. This is the setting a
  controller needs: to plan heating for the next day, it must predict a whole day ahead without yet knowing what will
  happen. We look at horizons from 15 minutes up to the full 10 days.

**Why is the difference between the two so important?**
<details><summary>Click for answer</summary>

A model can look excellent one step ahead and still be useless for control. One-step prediction mostly rewards "the
temperature barely changes in 15 minutes", which is easy (we saw the strong inertia in Question 1). Multi-step
open-loop is the honest, harder test: small modelling errors and any missing physics (like solar gain) build up over
hours, which is exactly when the differences between the models show.
</details>

In [16]:
# Observer: one-step prediction with the hidden wall node carried forward.
@njit(cache=False)
def observer(F, G, U, Tm, n, ns):
    xhat = np.zeros((n, ns)); pred1 = np.zeros(n)
    for k in range(ns): xhat[0, k] = Tm[0]           # start all nodes at the first measured temperature
    pred1[0] = Tm[0]
    for t in range(1, n):
        xp = F @ xhat[t-1] + G @ U[t-1]              # predict one step ahead
        pred1[t] = xp[0]
        xhat[t] = xp.copy(); xhat[t, 0] = Tm[t]      # reset the indoor node to the measurement, keep the wall estimate
    return pred1, xhat

# Open-loop forecast: from each anchor time, run forward k steps on the model's own predictions.
# We collect squared error, absolute error and signed error for every lead time k, averaged over all anchors.
@njit(cache=False)
def lead_curve(F, G, U, Tm, xhat, Kmax, n, ns):
    sse = np.zeros(Kmax+1); sae = np.zeros(Kmax+1); sbe = np.zeros(Kmax+1); cnt = np.zeros(Kmax+1)
    for a in range(n):
        x = xhat[a].copy()                           # launch the forecast from the observer's state estimate at anchor a
        kmax = Kmax if a + Kmax < n else n - 1 - a
        for k in range(1, kmax + 1):
            x = F @ x + G @ U[a + k - 1]
            e = x[0] - Tm[a + k]
            sse[k] += e*e; sae[k] += abs(e); sbe[k] += e; cnt[k] += 1
    return sse, sae, sbe, cnt

@njit(cache=False)
def free_run(F, G, U, x0, n, ns):                    # single open-loop run over the whole window
    out = np.zeros(n); x = x0.copy(); out[0] = x[0]
    for t in range(1, n):
        x = F @ x + G @ U[t-1]; out[t] = x[0]
    return out

<a id="sec-metrics"></a>
## 5. How we score the forecasts (RMSE, MAE, bias)

For a set of predicted and measured temperatures we report three numbers (all in °C / K, since they are
temperature differences):

- **RMSE** (root-mean-square error): the typical error size, but it punishes big mistakes extra hard.
- **MAE** (mean absolute error): the average error size, easier to read and less sensitive to outliers.
- **Bias** (mean of predicted − measured): the *direction* of the error. Positive means the model runs too warm,
  negative too cold. A model can have a small RMSE but a worrying bias, so we always look at it too.

In [17]:
def metrics(pred, meas):
    e = pred - meas
    return np.sqrt(np.mean(e**2)), np.mean(np.abs(e)), np.mean(e)   # RMSE, MAE, bias

COLM = {"A": "#9467bd", "B": "#2ca02c", "C": "#ff7f0e", "persistence": "#7f7f7f"}

# A simple, honest baseline: "persistence" = assume the temperature stays at its last measured value.
# Any useful model must beat this.
def persistence_lead(meas, Kmax, n):
    sse = np.zeros(Kmax+1); sae = np.zeros(Kmax+1); sbe = np.zeros(Kmax+1); cnt = np.zeros(Kmax+1)
    for a in range(n):
        kmax = min(Kmax, n - 1 - a)
        for k in range(1, kmax + 1):
            e = meas[a] - meas[a + k]                 # forecast = value at the anchor, held constant
            sse[k] += e*e; sae[k] += abs(e); sbe[k] += e; cnt[k] += 1
    return sse, sae, sbe, cnt

Now we run the evaluation for both windows and collect everything we need for the plots and the table.

In [18]:
KMAX = 96 * 4                       # study lead times up to 4 days
curves, resid_freerun, freeruns, table_rows = {}, {}, {}, []

for wname, (lo, hi) in WINDOWS.items():
    d, meas, src = get_window(lo, hi); n = len(meas)
    curves[wname] = {}; resid_freerun[wname] = {}; freeruns[wname] = {}
    t_days = d.day_of_year.values + d.second_of_day.values / 86400.0
    freeruns[wname]["time"] = t_days; freeruns[wname]["meas"] = meas
    freeruns[wname]["solar"] = d.global_horizontal_solar_radiation.values

    # --- persistence baseline ---
    sse, sae, sbe, cnt = persistence_lead(meas, KMAX, n)
    curves[wname]["persistence"] = np.sqrt(sse[1:] / np.maximum(cnt[1:], 1))
    one = np.concatenate(([meas[0]], meas[:-1]))                       # one-step persistence = previous value
    fr = np.full(n, meas[0])                                          # full-window persistence = constant
    r1, m1, b1 = metrics(one[1:], meas[1:]); rfr, mfr, bfr = metrics(fr, meas)
    table_rows.append([wname, "persistence", r1, m1, b1,
                       np.sqrt(sse[96]/cnt[96]), sbe[96]/cnt[96], rfr, bfr])

    # --- the three models ---
    for m in ["A", "B", "C"]:
        A, B, cols = matrices(m, FITS[m]); F, G = discretize(A, B, DT); ns = A.shape[0]; U = U_of(cols, src)
        pred1, xhat = observer(F, G, U, meas, n, ns)                  # one-step predictions + state estimates
        r1, m1, b1 = metrics(pred1[1:], meas[1:])
        sse, sae, sbe, cnt = lead_curve(F, G, U, meas, xhat, KMAX, n, ns)
        curves[wname][m] = np.sqrt(sse[1:] / np.maximum(cnt[1:], 1))
        r24, b24 = np.sqrt(sse[96]/cnt[96]), sbe[96]/cnt[96]          # 24-hour-ahead
        fr = free_run(F, G, U, xhat[0].copy(), n, ns)                # full 10-day open-loop run
        rfr, mfr, bfr = metrics(fr, meas)
        freeruns[wname][m] = fr; resid_freerun[wname][m] = fr - meas
        table_rows.append([wname, m, r1, m1, b1, r24, b24, rfr, bfr])

table = pd.DataFrame(table_rows, columns=["Window", "Model", "1-step RMSE", "1-step MAE", "1-step Bias",
                                          "24h RMSE", "24h Bias", "10-day RMSE", "10-day Bias"])
print("done")

done


<a id="sec-curves"></a>
## 6. Error vs. forecast horizon

The most informative single picture: how the typical error (RMSE) grows as we forecast further ahead, for each model and for the persistence baseline. One panel per window.

In [19]:
lead_h = np.arange(1, KMAX + 1) * 0.25                # lead time in hours
fig = make_subplots(rows=1, cols=2, subplot_titles=list(WINDOWS.keys()), shared_yaxes=True)
for j, wname in enumerate(WINDOWS, start=1):
    for m in ["persistence", "A", "B", "C"]:
        c = curves[wname][m]
        fig.add_trace(go.Scatter(x=lead_h, y=c, name=m if j == 1 else None,
                                 line=dict(color=COLM[m], width=2, dash="dot" if m == "persistence" else "solid"),
                                 legendgroup=m, showlegend=(j == 1)), 1, j)
    fig.update_xaxes(title_text="Forecast horizon [hours ahead]", row=1, col=j)
fig.update_yaxes(title_text="RMSE [K]", row=1, col=1)
fig.update_layout(title="Forecast error vs. how far ahead we predict")
save_fig(fig, "fig12_horizon_curves", h=440)

Two things stand out. **At very short horizons (left edge) all the curves sit on top of each other** — even
persistence. Over 15–60 minutes the temperature barely moves, so almost anything predicts it well and the models add
little. **As we look further ahead the models separate**, and Model C stays lowest, especially in summer.

Notice the curves are **not smoothly increasing** — they rise to a bump around 6–12 hours and dip again near 24 hours.

**Why doesn't the error just grow with the horizon?**
<details><summary>Click for answer</summary>

Because the indoor temperature has a strong 24-hour rhythm (we saw this in Question 1). Forecasting 6 hours ahead means
predicting from, say, a cool morning into a warm afternoon — a big change, so the error is large. Forecasting exactly 24
hours ahead lands at the *same time of day*, where conditions are similar again, so the error partly cancels. The error
curve therefore follows the daily cycle instead of rising straight up. This is also why persistence looks deceptively
good at exactly 24 hours.
</details>

<a id="sec-freerun"></a>
## 7. Forecast vs. reality: the 10-day open-loop run

The hardest test: start each model once at the beginning of the window and let it run for the full 10 days on its own predictions, with no measurement feedback. The black line is reality.

In [20]:
fig = make_subplots(rows=2, cols=1, subplot_titles=list(WINDOWS.keys()), vertical_spacing=0.12)
for r, wname in enumerate(WINDOWS, start=1):
    t = freeruns[wname]["time"]
    fig.add_trace(go.Scatter(x=t, y=freeruns[wname]["meas"], name="Measured" if r == 1 else None,
                             line=dict(color="black", width=2), legendgroup="m", showlegend=(r == 1)), r, 1)
    for m in ["A", "B", "C"]:
        fig.add_trace(go.Scatter(x=t, y=freeruns[wname][m], name=f"Model {m}" if r == 1 else None,
                                 line=dict(color=COLM[m], width=1.3), legendgroup=m, showlegend=(r == 1)), r, 1)
    fig.update_yaxes(title_text="Indoor T [°C]", row=r, col=1)
fig.update_xaxes(title_text="Day of year", row=2, col=1)
fig.update_layout(title="10-day open-loop forecast vs. measured")
save_fig(fig, "fig13_freerun_windows", h=640)

This is where the models part ways. **Models A and B drift away** from reality: in winter they run too warm, in
summer too cool. **Model C drifts far less and tracks reality best in both seasons** (in winter it, too, creeps slowly upward over many days, but only mildly compared with A and B; in summer it follows the daily peaks). The summer panel shows the difference most clearly —
A and B have no way to represent solar heating, so they miss the daytime warm-ups completely, while Model C follows
them.

<a id="sec-resid"></a>
## 8. Residuals: size, bias, and what the models miss

Residual = predicted − measured for the 10-day open-loop run. The spread shows the error size; the position relative to zero shows the bias (too warm above 0, too cool below).

In [21]:
fig = make_subplots(rows=1, cols=2, subplot_titles=list(WINDOWS.keys()), shared_yaxes=True)
for j, wname in enumerate(WINDOWS, start=1):
    for m in ["A", "B", "C"]:
        fig.add_trace(go.Violin(y=resid_freerun[wname][m], name=f"Model {m}" if j == 1 else None,
                                line_color=COLM[m], legendgroup=m, showlegend=(j == 1),
                                meanline_visible=True, points=False), 1, j)
    fig.add_hline(y=0, line=dict(color="grey", dash="dash"), row=1, col=j)
    fig.update_xaxes(title_text=wname, row=1, col=j)
fig.update_yaxes(title_text="Residual (predicted − measured) [K]", row=1, col=1)
fig.update_layout(title="Distribution of 10-day open-loop residuals")
save_fig(fig, "fig14_residual_violins", h=440)

The bias jumps out: in winter A and B sit clearly **above zero** (too warm), in summer clearly **below zero** (too
cool). Model C stays centred near zero in both seasons. To see *why* A and B miss the summer, the next plot puts their
summer error against the amount of sunshine.

In [22]:
# Summer diagnostic: open-loop residual vs. solar radiation. If a model lacks a solar term,
# its error should grow with the sun. We use the 24-hour-ahead residual so it is a realistic forecast error.
wname = "Summer (year 2, days 200–209)"
d, meas, src = get_window(200, 210); n = len(meas)
solar = d.global_horizontal_solar_radiation.values
fig = go.Figure()
for m in ["A", "B", "C"]:
    A, B, cols = matrices(m, FITS[m]); F, G = discretize(A, B, DT); ns = A.shape[0]; U = U_of(cols, src)
    _, xhat = observer(F, G, U, meas, n, ns)
    res24 = np.full(n, np.nan)
    for a in range(n - 96):                                  # 24-hour-ahead residual at each anchor
        x = xhat[a].copy()
        for k in range(96):
            x = F @ x + G @ U[a + k]
        res24[a + 96] = x[0] - meas[a + 96]
    ok = ~np.isnan(res24)
    fig.add_trace(go.Scatter(x=solar[ok], y=res24[ok], mode="markers",
                             marker=dict(size=3, opacity=0.4, color=COLM[m]), name=f"Model {m}"))
fig.add_hline(y=0, line=dict(color="grey", dash="dash"))
fig.update_layout(title="Summer: 24-hour-ahead error vs. sunshine",
                  xaxis_title="Solar radiation [W/m²]", yaxis_title="Residual (predicted − measured) [K]")
save_fig(fig, "fig15_summer_resid_vs_solar", h=440)

Models A and B (no solar term) drift more and more **too cool** as the sun gets stronger — their error is tied to
the sunshine they cannot see. Model C's error stays flat around zero, because its solar term accounts for that heat.
This is the clearest single explanation for why Model C wins in summer.

<a id="sec-table"></a>
## 9. Summary table

All numbers are out-of-sample (year 2), in K. Three horizons: one-step (15 min), 24-hours-ahead open-loop, and the full 10-day open-loop run. Lower RMSE/MAE is better; bias near zero is better.

In [23]:
def show(window):
    t = table[table.Window == window].drop(columns="Window").set_index("Model").round(3)
    return t

print("WINTER (year 2, days 1–10)")
show("Winter (year 2, days 1–10)")

WINTER (year 2, days 1–10)


,1-step RMSE,1-step MAE,1-step Bias,24h RMSE,24h Bias,10-day RMSE,10-day Bias
Model,,,,,,,
persistence,0.842,0.544,0.001,0.771,-0.054,2.418,-1.789
A,0.826,0.537,0.024,2.327,2.087,8.573,7.417
B,0.815,0.534,0.029,2.052,1.646,5.903,5.063
C,0.737,0.525,0.041,1.812,0.730,2.468,1.795


In [24]:
print("SUMMER (year 2, days 200–209)")
show("Summer (year 2, days 200–209)")

SUMMER (year 2, days 200–209)


,1-step RMSE,1-step MAE,1-step Bias,24h RMSE,24h Bias,10-day RMSE,10-day Bias
Model,,,,,,,
persistence,0.303,0.127,-0.004,0.811,-0.452,2.711,-1.993
A,0.299,0.125,-0.008,0.940,-0.726,3.247,-2.622
B,0.297,0.132,-0.015,1.210,-0.899,3.170,-2.460
C,0.269,0.120,0.020,0.570,0.404,0.952,0.767


<a id="sec-answer"></a>
## 10. Answer: which model is best, and why

**There is no single "best" model — it depends on what you need the forecast for.** Judging on out-of-sample
(year-2) error, the picture is consistent across both windows:

**For short-horizon prediction (15 minutes to ~1 hour ahead)** — e.g. a state estimator that always has fresh
measurements — **all three models are essentially tied, and even the persistence baseline is competitive** (one-step
RMSE ≈ 0.8 K in winter, ≈ 0.3 K in summer for every option). At this horizon the building's inertia does the work, so the
extra complexity of B and C buys almost nothing. If short-term prediction were the only goal, the simplest model (A) — or
just persistence — would be the sensible choice.

**For control-oriented multi-step forecasting (several hours to a day or more ahead)** — what an energy-management
controller actually needs — **Model C is clearly best**, and it is the only model that is **robust across both seasons**.
At 24-hours-ahead its RMSE is the lowest in both windows, and in the 10-day open-loop run it drifts by far the least — A and B run away from reality (badly so in winter), while Model C stays much closer (on a par with persistence in winter, well ahead of everything in summer).

**Why Model C, specifically:**
- **Horizon behaviour:** A and B are fine for a few minutes but their error and bias grow strongly over hours; C stays
  the lowest at every horizon beyond the first.
- **Robustness across seasons:** A and B break in summer — they run too cool because they have no solar term (the
  residual-vs-sunshine plot shows their error rising with the sun). C handles both winter and summer.
- **Bias:** A and B carry a large bias in the open-loop run (too warm in winter, too cool in summer); C stays close to
  unbiased.
- **Out-of-sample, not just training:** C wins on *year-2* data, not only on the training year. So its extra solar term
  is capturing real, useful physics, not just fitting noise.

**The important caveat (why we don't just crown the lowest-RMSE model).** From Question 2 we know that not all of
Model C's parameters are individually trustworthy: several of them ran into the edge of their allowed range, and its fit
is the worst-conditioned of the three. It performs best here because the *one* piece of physics it adds — solar gain —
genuinely matters, especially in summer. But its detailed individual parameters should not be over-interpreted as real
building properties. If we needed a model
whose parameters are all trustworthy, or the simplest model that is "good enough", **Model B** is a defensible choice in
winter, where solar matters little. And at very short horizons, persistence is a hard baseline to beat.

**Bottom line:** for the energy-management use that motivates this task — predicting a day or more ahead, in any season,
to plan heating — **Model C is the best choice**, because only it captures solar gain and stays accurate and unbiased
across both the winter and summer windows. For short-horizon prediction the choice barely matters, and a simpler, more
trustworthy model is preferable.